# 📌 ReAct

![Topic](https://img.shields.io/badge/Topic-ReAct-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-Agentic-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-July%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — ReAct (Reason + Act) is a prompting technique that makes a language model alternate between writing out its reasoning ("Thought") and taking an action in the world ("Action"), then reading the result ("Observation") before reasoning again. This loop lets the model plan, use tools, correct course, and ground its answers in real data instead of guessing from memory alone.</span>

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10+ |
| Libraries | `pip install google-generativeai` |


---
## 1. Overview

Before ReAct, researchers mostly treated two model behaviors separately. On one side there was chain of thought prompting, where a model reasons step by step but never actually touches the outside world, so it can drift into confident but wrong answers ("hallucination"). 

On the other side there was action generation, where a model calls tools or navigates an environment but does very little explicit reasoning about why.

ReAct's insight was to combine them. The paper, published by Yao, Zhao, Yu, Du, Shafran, Narasimhan and Cao, describes it as a way to generate both reasoning traces and task-specific actions in an interleaved manner, so reasoning traces help the model track and update its plan while actions let it pull in fresh information from outside sources. The authors tested it on question answering, fact-checking, and interactive tasks like text-based games and web navigation, and found it outperformed baselines while also being easier for humans to interpret and trust.

---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

The core loop has three repeating parts: Thought, Action, Observation. The model writes a thought (its reasoning about what to do next), picks an action (like searching Wikipedia or calling a calculator), receives an observation (the actual result), and then loops back to think again with that new information in hand. It keeps cycling until it has enough to give a final answer.


### 2.1 User question comes in 
The model receives a task or query, just like normal prompting.
### 2.2 Thought 
The model writes a short piece of reasoning in plain text, deciding what information it still needs or what to try next.

### 2.3 Action 
Based on that thought, the model emits a structured action, such as searching a knowledge base, calling an API, or executing a calculator function.

### 2.4 Observation 
The environment or tool returns a real result, which gets appended to the model's context.

### 2.5 Loop or stop
The model reads the observation, writes a new thought, and either takes another action (if it still lacks information) or produces a final answer once it has enough to respond confidently.

![ReAct.png](../assets/ReAct.png)

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Less hallucination** | Less hallucination than pure chain-of-thought, since claims can be checked against real retrieved data rather than only the model's internal knowledge. |
| 🟢 | **More interpretability** | More interpretable than plain action-only agents, because you can read the thoughts and see why each action was taken. |
| 🟢 | **Flexibility on long-horizon tasks** | More flexible on long-horizon tasks, since the model can adjust its plan mid-way if an observation is unexpected. |
| 🟢 | **Error recovery** | A bad tool result can trigger a new thought that tries a different action rather than the model just continuing blindly. |
| 🔴 | **Token Cost** | More tokens and latency than a single-shot answer, since each cycle adds a thought, action, and observation to the context. |
| 🔴 | **Tool Quality** | Still depends on tool quality, if the search or API returns bad data, the reasoning built on top of it can still go wrong. |
| 🔴 | **Can loop unnecessarily** | A model may keep taking actions when it already had enough information, wasting calls. |
| 🔴 | **Prompt design sensitivity** | Small changes in how examples are formatted can noticeably change how well the model follows the pattern. |
| 🔴 | **Harder to evaluate** | Success depends on the whole trace of thoughts and actions rather than just a final answer, which complicates automated grading. |

---
## 4. Code Example

> **Goal:** Minimal ReAct (Reason + Act) loop example using Google's Gemini API.

In [ ]:
import os
import re
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
model = genai.GenerativeModel("gemini-3.5-flash")

# A toy "tool" the model can call. In a real system this could be a
# web search, a database query, a calculator, etc.
def fake_search(query: str) -> str:
    fake_knowledge_base = {
        "capital of australia": "Canberra is the capital of Australia.",
        "tallest mountain": "Mount Everest is the tallest mountain above sea level.",
    }
    for key, value in fake_knowledge_base.items():
        if key in query.lower():
            return value
    return "No result found."

REACT_SYSTEM_PROMPT = """
You are an assistant that solves problems using the ReAct method.
At each step, respond in exactly this format:

Thought: <your reasoning about what to do next>
Action: search[<search query>]  OR  Action: finish[<final answer>]

Only output one Thought and one Action per turn. Wait for an
Observation before continuing.
"""

def run_react_loop(question: str, max_steps: int = 5) -> str:
    transcript = f"{REACT_SYSTEM_PROMPT}\nQuestion: {question}\n"

    for step in range(max_steps):
        response = model.generate_content(transcript)
        text = response.text.strip()
        print(f"--- Step {step + 1} ---\n{text}\n")

        transcript += text + "\n"

        # Check if the model wants to finish
        finish_match = re.search(r"Action:\s*finish\[(.*?)\]", text, re.DOTALL)
        if finish_match:
            return finish_match.group(1).strip()

        # Otherwise, look for a search action
        search_match = re.search(r"Action:\s*search\[(.*?)\]", text, re.DOTALL)
        if search_match:
            query = search_match.group(1).strip()
            observation = fake_search(query)
            transcript += f"Observation: {observation}\n"
        else:
            # Model didn't follow the format, stop to avoid an infinite loop
            break

    return "Could not reach a final answer within the step limit."


if __name__ == "__main__":
    answer = run_react_loop("What is the capital of Australia?")
    print("Final answer:", answer)

--- Step 1 ---
Thought: I need to find the capital of Australia. I will search for "capital of Australia".
Action: search[capital of Australia]

--- Step 2 ---
Thought: The observation confirms that Canberra is the capital of Australia. I can now finish and provide the answer.
Action: finish[Canberra]

Final answer: Canberra


---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **ReAct interleaves reasoning and acting instead of treating them separately.**
- **The Thought, Action, Observation cycle grounds answers in real data.**
- **It reduces hallucination compared to reasoning alone.**
- **It costs more tokens and latency than a single-shot answer.**
- **The pattern generalizes across question answering, games, and web navigation.**

</div>